## BertMorphTagger multiplicity

Theoretically, the model doesn't work well with lemma multiplicity ("teod": "tigu" / "tegu").

Testing how many words are unresolved after using VabamorfWithBertTagger on Estonian treebank (2.18).


In [9]:
import estnltk
from estnltk import Span, Layer, Text
import json
import pandas as pd
from tqdm import tqdm
from collections import Counter

from vabamorfwithberttagger import VabamorfWithBertTagger

## Data

In [3]:
with open("UD_Estonian-EDT-r2.18/et_edt-ud-train.conllu", "r", encoding="utf-8") as infile:
    train_data = infile.readlines()
train_data = [line.split("=")[1].strip() for line in train_data if line.startswith("# text")]

with open("UD_Estonian-EDT-r2.18/et_edt-ud-test.conllu", "r", encoding="utf-8") as infile:
    test_data = infile.readlines()
test_data = [line.split("=")[1].strip() for line in test_data if line.startswith("# text")]

with open("UD_Estonian-EDT-r2.18/et_edt-ud-dev.conllu", "r", encoding="utf-8") as infile:
    dev_data = infile.readlines()
dev_data = [line.split("=")[1].strip() for line in dev_data if line.startswith("# text")]

all_data = train_data + test_data + dev_data
len(all_data)

30929

In [4]:
def count_mitmesus(data, vmwbtagger):
    total_count = 0
    mitmesed = 0
    sonad = []
    for line in tqdm(data):
        text = Text(line)
        vmwbtagger.tag(text)
        for elem in text["morph_analysis"]:
            total_count += 1
            if len(list(elem.lemma)) > 1:
                mitmesed += 1
                sonad.append((elem.text, list(elem.lemma)))
        
    return total_count, mitmesed, sonad

## VabamorfWithBertTagger with post analysis

In [14]:
vbt = VabamorfWithBertTagger(use_postanalysis=True)

In [22]:
total, mitmesed, examples = count_mitmesus(all_data, vbt)

100%|████████████████████████████████████| 30929/30929 [29:29<00:00, 17.48it/s]


In [16]:
print("total:", total)
print("mitmesed:", mitmesed)
print("mitmeste %", round(mitmesed*100/total, 2))

total: 438407
mitmesed: 11029
mitmeste % 2.52


In [19]:
with open("VabamorfWithBertTagger_use_postanalysis_true_mitmesus_examples.json", "w", encoding="utf-8") as outf:
    json.dump(examples, outf, ensure_ascii=False)

In [7]:
with open("VabamorfWithBertTagger_use_postanalysis_true_mitmesus_examples.json", encoding="utf-8") as f:
    dd = json.load(f)

examples = [tuple(x) for x in dd]

In [17]:
print(len(examples))
examples[:10]

11029


[('Ekspressi', ['Ekspress', 'Ekspress']),
 ('summade', ['summ', 'summa']),
 ('jõudnud', ['jõudma', 'jõudnu', 'jõudnud', 'jõudnud', 'jõudnud']),
 ('Kuusmann', ['Kuusmann', 'Kuusmann']),
 ('olnuks', ['olnud', 'olema', 'olnu']),
 ('nende', ['see', 'tema']),
 ('Kuusmann', ['Kuusmann', 'Kuusmann']),
 ('Ekspressi', ['Ekspress', 'Ekspress']),
 ('Bankiga', ['Bank', 'Banki']),
 ('pea', ['pea', 'pea', 'pidama', 'pea'])]

In [5]:
examples[-10:]

[('nende', ['see', 'tema']),
 ('alusel', ['alune', 'alus']),
 ('alusel', ['alune', 'alus']),
 ('väikeste', ['väike', 'väikene']),
 ('Tõenäose', ['Tõenäone', 'Tõenäose', 'Tõenäose', 'tõenäone']),
 ('visualiseeru', ['visualiseeru', 'visualiseeru', 'visualiseeruma']),
 ('esine', ['esine', 'esine', 'esinema']),
 ('too', ['too', 'tooma']),
 ('kulu', ['kulg', 'kulu']),
 ('peitu', ['peit', 'peituma', 'peit'])]

In [8]:
print(*(t for t in examples if "Päikese" in t[0]), sep="\n")

('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['päike', 'päikene'])
('Päikesel', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['päike', 'päikene'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['päike', 'päikene'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['Päike', 'Päikene', 'Päikes', 'Päikese'])
('Päikese', ['päike', 'päikene'])
('Päikese', ['Päike', 'Päikene', 'Päik

In [13]:
# top n words, that have multiplicity
n = 20
freq = Counter(t[0] for t in examples)
print(*freq.most_common(n), sep="\n")

('nende', 562)
('neid', 360)
('neist', 152)
('olnud', 120)
('neile', 114)
('neil', 104)
('alusel', 97)
('Nende', 97)
('Liidu', 85)
('kuni', 83)
('taha', 75)
('sama', 71)
('saanud', 69)
('näinud', 62)
('Saksa', 58)
('pea', 56)
('Riigikogu', 49)
('Päikese', 48)
('kätte', 44)
('aluseks', 42)


## VabamorfWithBertTagger without post analysis

In [21]:
vbt2 = VabamorfWithBertTagger(use_postanalysis=False)

In [22]:
total2, mitmesed2, examples2 = count_mitmesus(all_data, vbt2)

100%|████████████████████████████████████| 30929/30929 [29:29<00:00, 17.48it/s]


In [23]:
print("total:", total2)
print("mitmesed:", mitmesed2)
print("mitmeste %", round(mitmesed2*100/total2, 2))

total: 438407
mitmesed: 11034
mitmeste % 2.52


In [24]:
print(len(examples2))
examples2[:10]

11034


[('Ekspressi', ['Ekspress', 'Ekspress']),
 ('summade', ['summ', 'summa']),
 ('jõudnud', ['jõudma', 'jõudnu', 'jõudnud', 'jõudnud', 'jõudnud']),
 ('Kuusmann', ['Kuusmann', 'Kuusmann']),
 ('olnuks', ['olnud', 'olema', 'olnu']),
 ('nende', ['see', 'tema']),
 ('Kuusmann', ['Kuusmann', 'Kuusmann']),
 ('Ekspressi', ['Ekspress', 'Ekspress']),
 ('Bankiga', ['Bank', 'Banki']),
 ('pea', ['pea', 'pea', 'pidama', 'pea'])]

In [25]:
with open("VabamorfWithBertTagger_use_postanalysis_false_mitmesus_examples.json", "w", encoding="utf-8") as outf:
    json.dump(examples2, outf, ensure_ascii=False)